# Ollama Model Inference

> Load and run ollama models


In [ ]:
# | default_exp llm


In [ ]:
# | exporti
import dotenv
import json
import os
import httpx

In [ ]:
# | exporti
dotenv.load_dotenv()


In [ ]:
# | exporti
from typing import Literal, Optional
from ollama import list as ollama_list


In [ ]:
# | hide
from nbdev.showdoc import *


In [ ]:
# | export
def list_local_models():
  """List local models."""
  return [m.model for m in ollama_list()["models"]]


In [ ]:
list_local_models()


In [ ]:
# | exporti
from tqdm import tqdm
from ollama import pull


In [ ]:
# | exporti
def _download_model(model_name: str):
  """Download a model from ollama."""
  current_digest, bars = "", {}
  for progress in pull(model_name, stream=True):
    digest = progress.get("digest", "")
    if digest != current_digest and current_digest in bars:
      bars[current_digest].close()

    if not digest:
      print(progress.get("status"))
      continue

    if digest not in bars and (total := progress.get("total")):
      bars[digest] = tqdm(total=total, desc=f"pulling {digest[7:19]}", unit="B", unit_scale=True)

    if completed := progress.get("completed"):
      bars[digest].update(completed - bars[digest].n)

    current_digest = digest


In [ ]:
model_name = "gemma3:27b"
local_models = list_local_models()
if model_name not in local_models:
  _download_model(model_name)


In [ ]:
# | exporti
from ollama import chat as ollama_chat


In [ ]:
ollama_response = ollama_chat(
  model=model_name, think=None, messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}]
)
ollama_response.message.content, ollama_response.message.thinking


In [ ]:
# | export
def create_message(content: Optional[str], role: Literal["user", "assistant", "system"] = "user"):
  """Create a message for the chat."""
  return {"role": role, "content": content}


In [ ]:
# | exporti
import base64
import mimetypes
from pathlib import Path


In [ ]:
# | export
def create_message_with_resource(
  text: str,
  resource_path: Path | str,
  role: Literal["user", "assistant"] = "user",
) -> dict:
  """Create a message with an attached file (image/PDF) for vision models.
  
  Args:
    text: The text prompt to accompany the resource
    resource_path: Path to the file (image or PDF)
    role: Message role (user or assistant)
    
  Returns:
    Message dict in litellm vision format with base64-encoded content
  """
  resource_path = Path(resource_path)
  
  # Read and encode file
  file_bytes = resource_path.read_bytes()
  base64_data = base64.b64encode(file_bytes).decode("utf-8")
  
  # Detect MIME type
  mime_type, _ = mimetypes.guess_type(str(resource_path))
  if mime_type is None:
    # Default based on extension
    ext = resource_path.suffix.lower()
    mime_map = {
      ".pdf": "application/pdf",
      ".png": "image/png",
      ".jpg": "image/jpeg",
      ".jpeg": "image/jpeg",
      ".gif": "image/gif",
      ".webp": "image/webp",
    }
    mime_type = mime_map.get(ext, "application/octet-stream")
  
  # Create message with multimodal content
  return {
    "role": role,
    "content": [
      {"type": "text", "text": text},
      {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{base64_data}"},
      },
    ],
  }


# Tool Calling

Utilities to convert typed Python functions into OpenAI-compatible tool schemas,
and an agentic loop that lets the LLM call tools iteratively.

In [ ]:
# | exporti
import inspect
import re as _re
from typing import Callable, get_type_hints


def _python_type_to_json(hint) -> str:
  """Map a Python type annotation to a JSON Schema type string."""
  _TYPE_MAP = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
  }
  origin = getattr(hint, "__origin__", None)
  if origin is list:
    return "array"
  return _TYPE_MAP.get(hint, "string")


def _parse_param_docs(docstring: str | None) -> dict[str, str]:
  """Extract parameter descriptions from a Google-style Args: section."""
  if not docstring:
    return {}
  result: dict[str, str] = {}
  in_args = False
  current_param: str | None = None
  current_desc_lines: list[str] = []

  for line in docstring.splitlines():
    stripped = line.strip()
    if stripped.lower().startswith("args:"):
      in_args = True
      continue
    if in_args:
      if stripped and not stripped[0].isspace() and not stripped.startswith("-") and ":" not in stripped:
        # Left-aligned non-param line → we've exited the Args section
        if not stripped[0].isalpha() or stripped.endswith(":"):
          break
      # Check for "param_name: description" or "param_name (type): description"
      m = _re.match(r"^(\w+)\s*(?:\([^)]*\))?\s*[:–—-]\s*(.+)", stripped)
      if m:
        if current_param:
          result[current_param] = " ".join(current_desc_lines).strip()
        current_param = m.group(1)
        current_desc_lines = [m.group(2)]
      elif current_param and stripped:
        current_desc_lines.append(stripped)
      elif not stripped and current_param:
        result[current_param] = " ".join(current_desc_lines).strip()
        current_param = None
        current_desc_lines = []

  if current_param:
    result[current_param] = " ".join(current_desc_lines).strip()
  return result

In [ ]:
# | export
def tool_from_callable(fn: Callable) -> dict:
  """Convert a typed Python function to an OpenAI tool schema dict.

  Extracts function name, docstring, type hints, and default values to produce
  the schema expected by OpenAI/OpenRouter tool calling APIs.

  Args:
    fn: A Python function with type annotations and optional docstring.

  Returns:
    A dict in OpenAI tool format: {"type": "function", "function": {...}}
  """
  sig = inspect.signature(fn)
  try:
    hints = get_type_hints(fn)
  except Exception:
    hints = {}

  doc = inspect.getdoc(fn) or ""
  description = doc.split("\n\n")[0].strip() if doc else fn.__name__
  param_docs = _parse_param_docs(doc)

  properties: dict[str, dict] = {}
  required: list[str] = []

  for name, param in sig.parameters.items():
    hint = hints.get(name)
    prop: dict = {"type": _python_type_to_json(hint) if hint else "string"}
    if name in param_docs:
      prop["description"] = param_docs[name]
    properties[name] = prop
    if param.default is inspect.Parameter.empty:
      required.append(name)

  schema: dict = {
    "type": "function",
    "function": {
      "name": fn.__name__,
      "description": description,
      "parameters": {
        "type": "object",
        "properties": properties,
      },
    },
  }
  if required:
    schema["function"]["parameters"]["required"] = required
  return schema

In [ ]:
# | exporti
def _chat_one(model: str, message: str, think: Optional[bool] = None, **kwargs):
  """Chat with the model."""
  return ollama_chat(
    model=model, think=think, messages=[create_message(message)], **kwargs
  ).message.content


In [ ]:
_chat_one("gemma3:27b", "Merhaba. Nasılsın?")


In [ ]:
# | exporti
from pydantic import BaseModel


In [ ]:
# | export
class OutputSchema(BaseModel):
  """Base class for structured LLM outputs. Subclass this to define your schema.

  The optional `content` field will be used for chat history when present."""

  content: str | None = None

In [ ]:
class _Response(OutputSchema):
  """Example response schema."""

  reasoning: str
  is_happy: bool


In [ ]:
_Response.model_validate_json(
  _chat_one(model_name, "Mulu musun?", think=None, format=_Response.model_json_schema())
)


# LLM Cloud


In [ ]:
# | exporti
from ollama import ChatResponse
import litellm
litellm.suppress_debug_info = True  # Stop printing "Provider List" spam
from litellm import completion as litellm_completion
from urllib.request import Request, urlopen


def _schema_name(schema: type[BaseModel]) -> str:
  name = getattr(schema, "__name__", "structured_response")
  return "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in name) or "structured_response"


def _openrouter_response_format(schema: type[BaseModel]) -> dict:
  return {
    "type": "json_schema",
    "json_schema": {
      "name": _schema_name(schema),
      "strict": True,
      "schema": schema.model_json_schema(),
    },
  }

In [ ]:
model = "openrouter/google/gemini-2.5-flash-lite-preview-09-2025"


In [ ]:
response = litellm_completion(
  model=model,
  messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}],
)
response.choices[0].message.content


In [ ]:
response = litellm_completion(
  model=model,
  messages=[create_message("Hello, how are you?")],
  response_format=_Response,
)


In [ ]:
_Response.model_validate_json(response.choices[0].message.content)


In [ ]:
# | export
Tool = Callable | tuple[dict, Callable]  # auto-schema or (raw_schema, handler)


class LLM:
  """A class for interacting with an LLM."""

  def __init__(
    self,
    model_name: str,  # The name of the model to use
    system_prompt: str = "",  # The system prompt to use.
    think: Optional[bool] = None,  # Whether to enable thinking mode
    output_schema: Optional[
      type[BaseModel]
    ] = None,  # Schema for structured output
    tools: list[Tool] | None = None,  # Tool functions for agentic calling
    max_tool_rounds: int = 10,  # Max iterations of the tool loop
  ):
    if tools and output_schema:
      raise ValueError("Cannot use both tools and output_schema at the same time")

    self.model_name = model_name
    self.system_prompt = system_prompt
    self.output_schema = output_schema
    self.messages: list[dict] = [create_message(system_prompt, "system")]
    self.think = think
    self.max_tool_rounds = max_tool_rounds
    self.use_openrouter = "openrouter" in model_name

    # Process tools into schemas and handlers
    self._tool_schemas: list[dict] = []
    self._tool_handlers: dict[str, Callable] = {}
    if tools:
      for tool in tools:
        if isinstance(tool, tuple):
          schema, handler = tool
          self._tool_schemas.append(schema)
          self._tool_handlers[schema["function"]["name"]] = handler
        else:
          schema = tool_from_callable(tool)
          self._tool_schemas.append(schema)
          self._tool_handlers[tool.__name__] = tool

    if not self.use_openrouter:
      self._verify()

  def generate(self, message: str, **kwargs) -> str | OutputSchema:
    """One-shot generation. Does not maintain conversation history."""
    messages = [create_message(self.system_prompt, "system"), create_message(message, "user")]
    if self._tool_schemas:
      text, _ = self._run_tool_loop(messages, **kwargs)
      return text
    if self.use_openrouter:
      return self._cloud_call(messages, **kwargs)
    return self._local_call(messages, **kwargs)

  def chat(self, message: str, **kwargs) -> str | OutputSchema:
    """Multi-turn chat. Maintains conversation history."""
    self.messages.append(create_message(message, "user"))

    if self._tool_schemas:
      text, self.messages = self._run_tool_loop(list(self.messages), **kwargs)
      return text

    if self.use_openrouter:
      response = self._cloud_call(self.messages, **kwargs)
    else:
      response = self._local_call(self.messages, **kwargs)

    # Extract content for history: schema responses have .content, plain responses are strings
    if isinstance(response, BaseModel):
      self.messages.append(create_message(self._response_to_history_content(response), "assistant"))
    else:
      self.messages.append(create_message(response, "assistant"))
    return response

  def generate_with_resource(
    self,
    message: str,
    resource_path: Path | str,
    **kwargs,
  ) -> str | OutputSchema:
    """One-shot generation with an attached file (image/PDF).
    
    Only works with cloud models (openrouter). Does not maintain conversation history.
    
    Args:
      message: The text prompt to accompany the resource
      resource_path: Path to the file (image or PDF)
      **kwargs: Additional args passed to the API
      
    Returns:
      Model response as string or OutputSchema
    """
    if not self.use_openrouter:
      raise ValueError("generate_with_resource only works with cloud models (openrouter)")
    
    messages = [
      create_message(self.system_prompt, "system"),
      create_message_with_resource(message, resource_path, "user"),
    ]
    return self._cloud_call(messages, **kwargs)

  # --- Raw call methods (return full message dict for tool loop) ---

  def _local_call_raw(self, messages: list[dict], **kwargs) -> dict:
    """Call local Ollama and return normalized message dict."""
    response: ChatResponse = ollama_chat(
      model=self.model_name,
      think=self.think,
      messages=messages,
      tools=self._tool_schemas or None,
      **kwargs,
    )
    msg = response.message
    # Normalize Ollama ToolCall objects to OpenAI format
    tool_calls = None
    if msg.tool_calls:
      tool_calls = []
      for i, tc in enumerate(msg.tool_calls):
        tool_calls.append({
          "id": f"call_{i}",
          "type": "function",
          "function": {
            "name": tc.function.name,
            "arguments": tc.function.arguments,
          },
        })
    return {
      "role": "assistant",
      "content": msg.content,
      "tool_calls": tool_calls,
    }

  def _cloud_call_raw(self, messages: list[dict], **kwargs) -> dict:
    """Call cloud model and return the raw assistant message dict."""
    headers, payload = self._build_openrouter_request(messages, **kwargs)
    if self._tool_schemas:
      payload["tools"] = self._tool_schemas

    request = Request(
      "https://openrouter.ai/api/v1/chat/completions",
      data=json.dumps(payload).encode("utf-8"),
      headers=headers,
      method="POST",
    )
    with urlopen(request, timeout=120) as response:
      body = json.load(response)
    msg = body["choices"][0]["message"]
    # Normalize: ensure arguments is a dict
    if msg.get("tool_calls"):
      for tc in msg["tool_calls"]:
        args = tc["function"].get("arguments")
        if isinstance(args, str):
          tc["function"]["arguments"] = json.loads(args)
    return msg

  async def _acloud_call_raw(self, messages: list[dict], **kwargs) -> dict:
    """Async cloud call returning raw assistant message dict."""
    headers, payload = self._build_openrouter_request(messages, **kwargs)
    if self._tool_schemas:
      payload["tools"] = self._tool_schemas

    async with httpx.AsyncClient(timeout=120.0) as client:
      response = await client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
      )
      response.raise_for_status()
      body = response.json()
    msg = body["choices"][0]["message"]
    if msg.get("tool_calls"):
      for tc in msg["tool_calls"]:
        args = tc["function"].get("arguments")
        if isinstance(args, str):
          tc["function"]["arguments"] = json.loads(args)
    return msg

  # --- Tool execution and agentic loop ---

  def _execute_tool_call(self, tool_call: dict) -> str:
    """Dispatch a tool call to its handler, return result as string."""
    fn_name = tool_call["function"]["name"]
    fn_args = tool_call["function"]["arguments"]
    if isinstance(fn_args, str):
      fn_args = json.loads(fn_args)
    handler = self._tool_handlers.get(fn_name)
    if handler is None:
      return f"Error: unknown tool '{fn_name}'"
    try:
      result = handler(**fn_args)
      return str(result)
    except Exception as e:
      return f"Error calling {fn_name}: {e}"

  def _run_tool_loop(self, messages: list[dict], **kwargs) -> tuple[str, list[dict]]:
    """Call LLM in a loop, executing tool calls until a final text response."""
    for _ in range(self.max_tool_rounds):
      if self.use_openrouter:
        assistant_msg = self._cloud_call_raw(messages, **kwargs)
      else:
        assistant_msg = self._local_call_raw(messages, **kwargs)

      messages.append(assistant_msg)

      if not assistant_msg.get("tool_calls"):
        return assistant_msg.get("content") or "", messages

      for tc in assistant_msg["tool_calls"]:
        result = self._execute_tool_call(tc)
        messages.append({
          "role": "tool",
          "tool_call_id": tc["id"],
          "content": result,
        })

    raise RuntimeError(f"Tool loop exceeded {self.max_tool_rounds} rounds")

  async def _arun_tool_loop(self, messages: list[dict], **kwargs) -> tuple[str, list[dict]]:
    """Async tool loop. Supports both sync and async tool handlers."""
    import asyncio

    for _ in range(self.max_tool_rounds):
      if self.use_openrouter:
        assistant_msg = await self._acloud_call_raw(messages, **kwargs)
      else:
        assistant_msg = await asyncio.to_thread(self._local_call_raw, messages, **kwargs)

      messages.append(assistant_msg)

      if not assistant_msg.get("tool_calls"):
        return assistant_msg.get("content") or "", messages

      for tc in assistant_msg["tool_calls"]:
        fn_name = tc["function"]["name"]
        fn_args = tc["function"]["arguments"]
        if isinstance(fn_args, str):
          fn_args = json.loads(fn_args)
        handler = self._tool_handlers.get(fn_name)
        if handler is None:
          result = f"Error: unknown tool '{fn_name}'"
        else:
          try:
            if asyncio.iscoroutinefunction(handler):
              result = str(await handler(**fn_args))
            else:
              result = str(await asyncio.to_thread(handler, **fn_args))
          except Exception as e:
            result = f"Error calling {fn_name}: {e}"
        messages.append({
          "role": "tool",
          "tool_call_id": tc["id"],
          "content": result,
        })

    raise RuntimeError(f"Tool loop exceeded {self.max_tool_rounds} rounds")

  # --- Existing call methods (non-tool path) ---

  def _local_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call local Ollama model."""
    response: ChatResponse = ollama_chat(
      model=self.model_name,
      think=self.think,
      messages=messages,
      format=self.output_schema.model_json_schema() if self.output_schema else None,
      **kwargs,
    )
    return self._parse_response(response.message.content)

  def _cloud_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call cloud model."""
    return self._openrouter_call(messages, **kwargs)

  async def _acloud_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Async cloud call."""
    return await self._aopenrouter_call(messages, **kwargs)

  def _build_openrouter_request(self, messages: list[dict], **kwargs) -> tuple[dict, dict]:
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
      raise RuntimeError("OPENROUTER_API_KEY is not configured")

    headers = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json",
    }
    http_referer = kwargs.pop("http_referer", None) or os.getenv("OPENROUTER_HTTP_REFERER")
    x_title = kwargs.pop("x_title", None) or os.getenv("OPENROUTER_X_TITLE")
    if http_referer:
      headers["HTTP-Referer"] = http_referer
    if x_title:
      headers["X-Title"] = x_title

    provider = dict(kwargs.pop("provider", {}) or {})
    if self.output_schema:
      provider.setdefault("require_parameters", True)

    payload = {
      "model": self.model_name.removeprefix("openrouter/"),
      "messages": messages,
      **kwargs,
    }
    if provider:
      payload["provider"] = provider

    if self.think is True and "reasoning" not in payload:
      payload["reasoning"] = {"effort": "low"}
    if self.output_schema:
      payload["response_format"] = _openrouter_response_format(self.output_schema)
      if not payload.get("stream"):
        plugins = list(payload.get("plugins") or [])
        if not any(plugin.get("id") == "response-healing" for plugin in plugins if isinstance(plugin, dict)):
          plugins.append({"id": "response-healing"})
        payload["plugins"] = plugins

    return headers, payload

  def _parse_openrouter_body(self, body: dict) -> str | OutputSchema:
    return self._parse_response(body["choices"][0]["message"]["content"])

  def _openrouter_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    headers, payload = self._build_openrouter_request(messages, **kwargs)

    request = Request(
      "https://openrouter.ai/api/v1/chat/completions",
      data=json.dumps(payload).encode("utf-8"),
      headers=headers,
      method="POST",
    )
    with urlopen(request, timeout=120) as response:
      body = json.load(response)
    return self._parse_openrouter_body(body)

  async def _aopenrouter_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    headers, payload = self._build_openrouter_request(messages, **kwargs)
    async with httpx.AsyncClient(timeout=120.0) as client:
      response = await client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
      )
      response.raise_for_status()
      body = response.json()
    return self._parse_openrouter_body(body)

  async def agenerate(self, message: str, **kwargs) -> str | OutputSchema:
    """Async one-shot generation. Does not maintain conversation history."""
    import asyncio

    messages = [create_message(self.system_prompt, "system"), create_message(message, "user")]
    if self._tool_schemas:
      text, _ = await self._arun_tool_loop(messages, **kwargs)
      return text
    if self.use_openrouter:
      return await self._acloud_call(messages, **kwargs)
    return await asyncio.to_thread(self._local_call, messages, **kwargs)

  def _parse_response(self, content) -> str | OutputSchema:
    """Parse raw response content, validating against schema if set."""
    assert content is not None, "No content was returned"
    if self.output_schema:
      if isinstance(content, str):
        return self.output_schema.model_validate_json(content)
      return self.output_schema.model_validate(content)
    if isinstance(content, str):
      return content
    return json.dumps(content, ensure_ascii=False)

  def _response_to_history_content(self, response: BaseModel) -> str:
    content = getattr(response, "content", None)
    if isinstance(content, str) and content:
      return content
    return response.model_dump_json()

  async def agenerate_batch(
    self,
    messages: list[str],
    concurrency: int = 10,
    show_progress: bool = True,
    **kwargs,
  ) -> list[str | OutputSchema]:
    """Batch async generation with concurrency control.

    Args:
        messages: List of input messages to process
        concurrency: Max parallel requests (default 10)
        show_progress: Show tqdm progress bar
        **kwargs: Additional args passed to the API (e.g., temperature)

    Returns:
        List of responses in same order as inputs
    """
    import asyncio
    from tqdm.asyncio import tqdm_asyncio

    semaphore = asyncio.Semaphore(concurrency)

    async def process_one(message: str) -> str | OutputSchema:
      async with semaphore:
        return await self.agenerate(message, **kwargs)

    tasks = [process_one(msg) for msg in messages]
    if show_progress:
      return await tqdm_asyncio.gather(*tasks, desc="Processing")
    return await asyncio.gather(*tasks)

  def _verify(self):
    """Verify the model is available, download if missing."""
    if self.model_name not in list_local_models():
      _download_model(self.model_name)

# Tests


In [ ]:
# Test create_message with different roles
assert create_message("hello") == {"role": "user", "content": "hello"}
assert create_message("hello", "assistant") == {"role": "assistant", "content": "hello"}
assert create_message("system prompt", "system") == {"role": "system", "content": "system prompt"}
assert create_message(None) == {"role": "user", "content": None}
print("✓ create_message tests passed")


In [ ]:
# Test OutputSchema validation
class TestSchema(OutputSchema):
  sentiment: str

# Valid JSON should parse correctly
parsed = TestSchema.model_validate_json('{"content": "hello", "sentiment": "positive"}')
assert parsed.content == "hello"
assert parsed.sentiment == "positive"

# content field is optional
no_content = TestSchema.model_validate_json('{"sentiment": "positive"}')
assert no_content.content is None
assert no_content.sentiment == "positive"

print("✓ OutputSchema tests passed")

In [ ]:
# Test LLM initialization and routing logic
# OpenRouter models should set use_openrouter=True
llm_cloud = LLM("openrouter/google/gemini-2.5-flash", system_prompt="Be helpful")
assert llm_cloud.use_openrouter == True
assert llm_cloud.model_name == "openrouter/google/gemini-2.5-flash"
assert llm_cloud.system_prompt == "Be helpful"
assert llm_cloud.messages == [{"role": "system", "content": "Be helpful"}]

print("✓ LLM initialization tests passed")


In [ ]:
# Test list_local_models returns a list of strings
models = list_local_models()
assert isinstance(models, list)
assert all(isinstance(m, str) for m in models)
print(f"✓ list_local_models tests passed (found {len(models)} models)")


In [ ]:
# Test LLM._parse_response without schema (returns string as-is)
llm_no_schema = LLM("openrouter/test", output_schema=None)
assert llm_no_schema._parse_response("hello world") == "hello world"

# Test LLM._parse_response with schema
llm_with_schema = LLM("openrouter/test", output_schema=TestSchema)
result = llm_with_schema._parse_response('{"content": "test", "sentiment": "neutral"}')
assert isinstance(result, TestSchema)
assert result.content == "test"
assert result.sentiment == "neutral"

print("✓ LLM._parse_response tests passed")


In [ ]:
# Test agenerate_batch works with local Ollama models
import asyncio
async def test_agenerate_batch_local():
  llm_local = LLM("gemma3:27b")
  results = await llm_local.agenerate_batch(["Say 'test' and nothing else"], show_progress=False)
  assert isinstance(results, list)
  assert len(results) == 1
  return True

assert asyncio.run(test_agenerate_batch_local())
print("✓ agenerate_batch local model test passed")


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()
